# tiny-log-parser — v3 demo, real-log eval, and a messy-log head-to-head

A Qwen3-4B fine-tune that normalizes a messy log line into a canonical 7-field
JSON record, emitting `null` for fields the line does not carry instead of
inventing them.

Three things run here:

1. **The demo** (sections 2–6): the model on six synthetic formats, then on real
   production logs from LogHub, where most lines have no level and no trace id.
2. **The eval** (sections 7–10): four arms — deterministic rules, the v2
   fine-tune, the v3 fine-tune, `gemini-3.1-pro-preview` — scored against real
   lines with independently written labels.
3. **The messy head-to-head** (section 11): v3 vs v2 vs Gemini on 20 lines in
   seven formats that are **not** LogHub, where `rule_parser.py` has no coverage
   at all. No labels, no score — a side-by-side you read yourself.

**The results, so you know what you are re-running.** They point in opposite
directions on the two corpora, and that is the honest state of this project:

| corpus | rules | gemini | v2 | v3 |
|---|---|---|---|---|
| test, n=127, labels share an author with the parser | 92.1% | **96.1%** | 73.2% | not run |
| P1, n=96, independently written labels | **84.4%** | 78.1% | 60.4% | **83.3%** |

On P1 v3 finishes ahead of Gemini (83.3% vs 78.1%) but **p = 0.0625** — only 5
discordant pairs exist, all favouring v3, and a two-sided exact test on n=5
floors at 0.0625, so significance was unreachable at this corpus size. It is
level with the rules parser (p = 1.0) and beats v2 22–0 (p < 0.0001).

**v3 does not beat `rule_parser.py`**, and was pre-registered not to: 75% of its
training labels are that parser's output, so the parser is its ceiling. See
`v3/PREREGISTRATION_P2.md`, written before the run.

Prediction files are committed, so sections 9 and 10 run with no GPU and no API
key. Section 11 needs both.

Outputs are cleared; this notebook is meant to be run, not read.
**Runtime → Change runtime type → T4 GPU.**

Repo: https://github.com/arshirazi97/tiny-log-parser

## 1. Setup (~3 min)

In [ ]:
import os
os.chdir('/content')
!rm -rf tiny-log-parser
!git clone -q https://github.com/arshirazi97/tiny-log-parser.git
os.chdir('/content/tiny-log-parser')
!pip install -q "transformers==4.51.3" "peft==0.20.0" accelerate bitsandbytes openai
!pip uninstall -q -y torchao     # Colab ships 0.10.0; peft 0.20 raises on anything under 0.16

import importlib.util, transformers, peft
stale = ((transformers.__version__, peft.__version__) != ("4.51.3", "0.20.0")
         or importlib.util.find_spec("torchao") is not None)
if stale:
    print("restarting to pick up the new versions...")
    os.kill(os.getpid(), 9)      # re-run this cell once the runtime comes back

`peft==0.20.0` matches the version that wrote `adapter_config.json`. Older peft
still loads the adapter, but warns and **silently ignores** the config keys it
does not recognise — not something to run an eval through.

The `torchao` uninstall is not incidental. peft's `is_torchao_available()`
**raises** when torchao is importable and below 0.16.0, and Colab preinstalls
0.10.0, so `PeftModel.from_pretrained` dies with an `ImportError` that has
nothing to do with this adapter. Absent, the check returns False and the load
proceeds; nothing here uses torchao, since quantization is bitsandbytes. The
restart guard covers the case where torchao was already imported — removing the
files does not retract it from `sys.modules`, so the cell restarts once and
comes back clean.

## 2. Load an adapter — v3 by default

Base weights (3.5 GB) plus a LoRA adapter (132 MB). v1, v2 and v3 are separate
HuggingFace repos, so every comparison in this project stays reproducible.
Change `ADAPTER` below to `...-v2` to run the whole demo as v2 instead.

In [ ]:
import torch, json, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from eval import parse
from schema_v2 import SPEC_EVAL, FIELDS
from score_hybrid import epoch_override

BASE    = 'unsloth/qwen3-4b-unsloth-bnb-4bit'
ADAPTER = 'arshirazi/tiny-log-parser-v3'   # or ...-v2 to re-run the v2 demo

tok = AutoTokenizer.from_pretrained(BASE, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE, device_map='auto'), ADAPTER).eval()

# decoding is greedy everywhere in this project; clear the sampling defaults so
# generate() stops warning about temperature/top_p it is not using
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print(f'ready | {ADAPTER} | SPEC_EVAL {len(SPEC_EVAL)} chars')

## 3. The pipeline
Two details that are easy to get wrong:

- **The chat template, not `eval.build_prompt`.** That helper builds the v1 flat
  prompt, which is not the format the adapter was trained through. The v1 demo
  used it; this does not.
- **`SPEC_EVAL`, not `SPEC`.** Same frozen training spec plus the three
  clarifications that labelling the real corpus forced (`ADJUDICATION.md`
  S1b/F5/M1b). Every arm gets the identical text.

The epoch pre-pass fires only on bare epoch integers — the one subtask the model
reliably fails.

In [ ]:
def build(raw):
    return tok.apply_chat_template(
        [{'role': 'system', 'content': SPEC_EVAL}, {'role': 'user', 'content': raw}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

def normalize(lines, batch=8):
    out = []
    for i in range(0, len(lines), batch):
        chunk = lines[i:i+batch]
        enc = tok([build(l) for l in chunk],
                  return_tensors='pt', padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=200, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for l, o in zip(chunk, gen):
            p = parse(tok.decode(o[enc.input_ids.shape[-1]:], skip_special_tokens=True))
            iso = epoch_override(l)
            if p and iso: p = {**p, 'timestamp': iso}
            out.append(p)
    return out

def show(lines, tags=None):
    t0 = time.time(); preds = normalize(lines); dt = time.time() - t0
    for j, (l, p) in enumerate(zip(lines, preds)):
        tag = f'  [{tags[j]}]' if tags else ('  [pre-pass]' if epoch_override(l) else '')
        print('\nIN  ', l[:96], tag)
        print('OUT ', json.dumps(p) if p else '[unparseable]')
    print(f'\n{len(lines)} lines in {dt:.1f}s ({dt/len(lines)*1000:.0f} ms/line)')
    return preds

## 4. Six synthetic formats, one schema
The distribution the model was trained on.

In [ ]:
_ = show([
  '<131>Mar  5 02:10:12 web-07 payments[4471]: TLS handshake aborted by peer [tid=8f2c91aa04bd77e35c1d6b0392ef4a18] took=4.775s',
  '10.14.2.9 - - [22/Jul/2026:09:15:44 +0500] "GET /api/v2/orders HTTP/1.1" 503 812 "-" "curl/8.4.0" rt=7.881',
  'ts=1780543196 level=warn service=inventory msg="stock below threshold" trace=7f3b1c2d4e5a6b8c9d0e1f2a3b4c5d6e latency=340ms',
  '2026-06-18 14:22:09,331 ERROR [payment-worker-3] c.a.p.RefundService - refund gateway timeout traceId=b2e4f6a8c0d2e4f6a8b0c2d4e6f8a0b2 elapsed=2140ms',
  '{"time":1783402811,"severity":"CRITICAL","container":"billing","message":"ledger write failed","duration_us":9120000}',
  '[2026-07-15T03:44:12+05:00] [FATAL] [search-indexer] shard replication halted (tid=e9d8c7b6a5f4e3d2c1b0a9f8e7d6c5b4) 15.2s',
])

Two things worth checking in that output. The access-log line has a 503 and no
level token — v2 should emit `level: null`, where v1 emitted `ERROR` inferred
from the status code. And `severity: CRITICAL` should normalize to `FATAL`,
which is in the enum; v1 passed `CRITICAL` straight through.

## 5. Real logs — the null v1 could not emit
Six lines from the dev corpus (LogHub: HDFS, OpenSSH, Linux, Proxifier,
HealthApp, Zookeeper). The first four carry **no level token at all**. The spec
is explicit that "authentication failure" is not an `ERROR` — wording is not a
level. The last two do carry one, so this is not a model that just always says
null.

In [ ]:
dev = {r['id']: r for r in map(json.loads, open('real-eval/corpus_dev.jsonl'))}
picks = ['4702105f24cc',   # OpenSSH  pam_unix authentication failure -- no level
         '88966aeef5ab',   # Linux    kernel isapnp scan          -- no level
         '6fdf6eb037e6',   # Proxifier bytes sent/received        -- no level, no latency
         '23f056a103ac',   # HealthApp pipe-delimited             -- no level
         '93418bafbdce',   # HDFS     INFO dfs.DataNode$PacketResponder
         '0245d562d619']   # Zookeeper INFO nested in a thread bracket

_ = show([dev[i]['raw'] for i in picks], tags=[dev[i]['source'] for i in picks])

Those `level` numbers were measured on **v2**: on the 50 dev lines it abstains
34% of the time and hallucinates a level on **0** of the 16 lines carrying no
level token, against v1's 0% and 100% — the gap that motivated the retrain
(`real-eval/probe_dev_v2.json`).

v3 holds that result and adds extraction. On the 96 P1 lines it hallucinated a
level on **0 of 32** gold-null opportunities, matched v2, and closed the two
families v2 failed: `latency_ms` errors 8 → **0**, non-disputed `service` errors
14 → **1**.

## 6. Why the epoch pre-pass exists
The model gets minutes and seconds right and the date wrong — epoch → calendar
arithmetic is integer division it cannot do. Scaling training data 5k → 20k
moved this 0.5 points, so it was routed to `datetime.fromtimestamp()` instead.

In [ ]:
line = 'ts=1780543196 level=warn service=inventory msg="stock below threshold" latency=340ms'
enc = tok(build(line), return_tensors='pt').to(model.device)
g = model.generate(**enc, max_new_tokens=200, do_sample=False, pad_token_id=tok.pad_token_id)
raw = parse(tok.decode(g[0][enc.input_ids.shape[-1]:], skip_special_tokens=True))
print('model alone :', raw.get('timestamp'))
print('pre-pass    :', epoch_override(line))

## 7. The real-log eval — smoke, then dev, then the sealed test set
`predict.py` writes `{id, pred}` per line for `score_arms.py`. Order matters:
the test corpus is scored **once**, which is why it refuses to load without
`--allow-test`.

Smoke first — 8 dev lines. Check `spec=SPEC_EVAL (3084 chars)` and
**0 unparseable** before spending GPU time on the rest.

In [ ]:
import os
FORCE = False   # True runs the 8-line smoke -- ~2 min, mostly model load

if FORCE or not os.path.exists('real-eval/preds_dev_model.jsonl'):
    !python real-eval/predict.py --arm model --corpus real-eval/corpus_dev.jsonl --limit 8
else:
    print('dev predictions are already in the repo -- smoke test skipped.')


Dev, 50 lines (~4 min). The scorer prints `PROVISIONAL` because 49 of the 50 dev
labels are still `_review: pending`, and rules score 98% on dev largely by
construction (`LABEL_REVIEW_TEST.md`) — neither number is a result. What to look
for is the model arm landing near **80% six-field**. A large gap means the prompt
path or the adapter is wrong.

(The earlier `probe_dev_v2.json` run scored 74% on these same lines. The
difference is `SPEC_EVAL`: the probe sent the frozen `SPEC` without the three
clarifications, and adding them moved message 40% → 56% and service 80% → 86%.)

In [ ]:
import os
FORCE = False   # True regenerates -- ~4 min of GPU

if FORCE or not os.path.exists('real-eval/preds_dev_model.jsonl'):
    !python real-eval/predict.py --arm model --corpus real-eval/corpus_dev.jsonl
else:
    print('real-eval/preds_dev_model.jsonl is already in the repo -- skipping.')
    print('Set FORCE = True above to regenerate it.')

!python real-eval/score_arms.py --labels real-eval/labels_dev.jsonl rules model=real-eval/preds_dev_model.jsonl


The sealed test set, 127 lines (~10 min). Run this once. Re-running it with
different decoding and keeping the better number spends the test set twice.

In [ ]:
import os
FORCE = False   # True regenerates -- ~10 min of GPU, and it is the sealed test set

if FORCE or not os.path.exists('real-eval/preds_test_model.jsonl'):
    !python real-eval/predict.py --arm model --corpus real-eval/corpus_test.jsonl --allow-test
else:
    print('real-eval/preds_test_model.jsonl is already in the repo -- skipping.')
    print('Set FORCE = True above to regenerate it.')


Only if you regenerated them above. The committed copies are already in the repo, so a fresh clone has them and there is nothing to download.

In [ ]:
# only meaningful if you set FORCE = True above
# from google.colab import files
# files.download('real-eval/preds_test_model.jsonl')
# files.download('real-eval/preds_dev_model.jsonl')
print('nothing to download -- predictions ship with the repo')


## 8. The Gemini arm
No GPU needed, just a key (openrouter.ai/keys). **Measured: $2.02 for the 127
test lines**, $0.0159/line at 8.8 s/line — Gemini spends ~1,103 completion tokens
per line, because reasoning tokens bill at output rate. Budget accordingly; a
short run first will print the exact per-line cost. The cell below uses
`corpus_demo.jsonl` -- the same six lines section 5 shows, four of them with no
level token -- so it costs about **$0.10** and actually exercises abstention. The
first five dev lines would be four HDFS and one Zookeeper, all carrying a level,
where both arms agree and nothing is learned.

One wrinkle: the baseline gets 3 few-shot examples — the same deliberate
handicap on the fine-tune as in the v1 comparison — and they come from
`train_v2.jsonl`, which is gitignored. Regenerating at `--train 200` is enough:
the generator is seeded, so the first 3 rows are byte-identical to the 20k
training run.

In [ ]:
import getpass, os

key = getpass.getpass('OpenRouter API key (press Enter to skip, ~$0.10 if you continue): ').strip()
if not key:
    print('skipped -- no key given. The committed Gemini predictions are still scored in section 9.')
else:
    os.environ['OPENROUTER_API_KEY'] = key
    # few-shot examples come from train_v2.jsonl, which is gitignored; the
    # generator is seeded, so the first 3 rows match the 20k training run
    !python generate_v2.py --train 200 --test 20
    !python real-eval/predict.py --arm gemini --corpus real-eval/corpus_demo.jsonl --out real-eval/preds_demo_gemini.jsonl


Confirm **0 unparseable** above before the real run. A wrong `--gemini-model`
slug fails four times per line and writes 127 nulls without stopping.

In [ ]:
import os
FORCE = False   # True regenerates -- $2.02 of API credit

if FORCE or not os.path.exists('real-eval/preds_test_gemini.jsonl'):
    !python real-eval/predict.py --arm gemini --corpus real-eval/corpus_test.jsonl --allow-test
else:
    print('real-eval/preds_test_gemini.jsonl is already in the repo -- skipping.')
    print('Set FORCE = True above to regenerate it.')

# download only if you regenerated it
# from google.colab import files; files.download('real-eval/preds_test_gemini.jsonl')


## 9. Score every arm

Two tables, because the two corpora disagree and both belong in the record.

**The 127-line test corpus.** The rules arm is inflated here: the labels and the
parser share an author. It was 100% by construction until the S2c correction
moved 10 Proxifier labels off the parser. Gemini takes six-field 96.1% against
v2's 73.2%, 29–0 on discordant pairs.

**The 96-line P1 corpus** has labels written independently of the parser's
rulebook, and it is where v3 is scored. The ordering reverses: rules 84.4%,
v3 83.3%, gemini 78.1%, v2 60.4%.

Read the McNemar block, not the percentages — it is paired, so it has real power
at these sample sizes. Watch for three things: `rules vs v3` at p = 1.0 (level,
not a win), `gemini vs v3` at p = 0.0625 (ahead, underpowered), and `v2 vs v3`
at p < 0.0001 (the actual finding).

**13 of the 96 lines measure a labelling dispute, not a model.** On those
Zookeeper lines `ADJUDICATION.md:84` says take the class before `@<line>`, and
the annotator labelled them null on a blind second pass
(`v3/PREREGISTRATION_AMENDED.md:621`). Every arm follows the written rule, so
every arm loses them. `subsets_p3.py` writes the label files that exclude them.

In [ ]:
# --- the 127-line test corpus: rules, v2, gemini -------------------------
TEST = ("rules model=real-eval/preds_test_model.jsonl "
        "gemini=real-eval/preds_test_gemini.jsonl")
!python real-eval/score_arms.py --labels real-eval/labels_test.jsonl {TEST}

# --- the 96-line P1 corpus, independent labels: all four arms ------------
P1 = ("rules gemini=real-eval/preds_p1_gemini.jsonl "
      "v2=real-eval/preds_p1_model.jsonl v3=real-eval/preds_p1_v3.jsonl")
!python real-eval/score_arms.py --labels real-eval/spotcheck_p1_adjudicated.jsonl {P1}

# --- and with the 13 disputed Zookeeper lines removed (n=83) -------------
!python real-eval/subsets_p3.py --write
!python real-eval/score_arms.py --labels real-eval/spotcheck_p1_adjudicated_nos1b.jsonl {P1}

## 10. Where v3 and Gemini actually disagree
Reads the committed prediction files — no GPU, no API cost. The 96 P1 lines,
gold in the first column, mismatches highlighted.

In [ ]:
import pandas as pd, json
from IPython.display import display, HTML

lab = {r['id']: r for r in map(json.loads, open('real-eval/spotcheck_p1_adjudicated.jsonl'))}
mine = {r['id']: r['pred'] for r in map(json.loads, open('real-eval/preds_p1_v3.jsonl'))}
gem  = {r['id']: r['pred'] for r in map(json.loads, open('real-eval/preds_p1_gemini.jsonl'))}

SIX = ['timestamp', 'level', 'service', 'trace_id', 'status_code', 'latency_ms']
shown = 0
for i, row in lab.items():
    a, b = mine.get(i) or {}, gem.get(i) or {}
    if all(a.get(f) == b.get(f) for f in SIX):
        continue
    df = pd.DataFrame({'gold':   [row['label'][f] for f in SIX],
                       'v3':     [a.get(f) for f in SIX],
                       'gemini': [b.get(f) for f in SIX]}, index=SIX)
    display(HTML(f"<pre style='margin:16px 0 4px;white-space:pre-wrap'>"
                 f"<b>{row['source']}</b>  {row['raw'][:120]}</pre>"))
    display(df.style.apply(
        lambda s: ['background-color:#fff3cd' if v != g else ''
                   for v, g in zip(s, df['gold'])], axis=0))
    shown += 1
    if shown == 8:
        break
print(f'{shown} disagreements shown')

---

## 11. Messy logs — v3 vs v2 vs Gemini, no labels

`messy.log` is 20 lines in seven shapes — syslog with a priority prefix, Apache
combined, logfmt, JSON, bracket-ISO, Java/logback, and one deliberately
truncated line. **None of them are LogHub formats**, and `rule_parser.py`
returns `None` on 19 of 20.

That is why this section exists. On P1 the parser is the ceiling and v3 is one
line short of it, so a model buys you nothing there. Here the parser has no
coverage, which is the only regime where a fine-tune can be worth its GPU.

**Run v2 as well as v3.** v2 trained on `generate_v2.py` synthetic data, which
produces shapes like these; v3's mix moved to 75% real LogHub plus two targeted
families. v3 may be *worse* here. That is a live possibility, not a
hypothetical, and it is one adapter swap to find out.

**There are no gold labels and none are invented.** Nothing below prints an
accuracy number. Two arms agreeing can both be wrong; a disagreement tells you
only that one of them is.

In [ ]:
# build the corpus and run the rules arm -- expect 19/20 unparseable
!python real-eval/messy_corpus.py messy.log -o real-eval/corpus_messy.jsonl
!python real-eval/predict.py --arm rules --corpus real-eval/corpus_messy.jsonl --out real-eval/preds_messy_rules.jsonl

To use your own lines instead, write them to a file and rebuild — one log line
per line, no labels needed:

```python
open('mine.log','w').write('''<paste your log lines here>''')
!python real-eval/messy_corpus.py mine.log -o real-eval/corpus_messy.jsonl
```

In [ ]:
# v3 and v2, same base, different adapters (~2 min each, mostly model load)
V3, V2 = 'arshirazi/tiny-log-parser-v3', 'arshirazi/tiny-log-parser-v2'

!python real-eval/predict.py --arm model --adapter {V3} --corpus real-eval/corpus_messy.jsonl --batch 32 --out real-eval/preds_messy_v3.jsonl
!python real-eval/predict.py --arm model --adapter {V2} --corpus real-eval/corpus_messy.jsonl --batch 32 --out real-eval/preds_messy_v2.jsonl

`--shots 0` below is deliberate. The labelled eval gives Gemini three few-shot
examples as a handicap against an untrained baseline; for a straight
head-to-head both arms should get the identical prompt — `SPEC_EVAL` and nothing
else. It also drops the dependency on `train_v2.jsonl`, which is gitignored and
absent from a fresh clone. 20 lines costs about **$0.02**.

In [ ]:
import getpass, os

key = getpass.getpass('OpenRouter API key (Enter to skip, ~$0.02 if you continue): ').strip()
if not key:
    print('skipped -- the next cell will compare v3 against v2 only.')
else:
    os.environ['OPENROUTER_API_KEY'] = key
    !python real-eval/predict.py --arm gemini --shots 0 --corpus real-eval/corpus_messy.jsonl --out real-eval/preds_messy_gemini.jsonl

In [ ]:
import os
arms = "v3=real-eval/preds_messy_v3.jsonl v2=real-eval/preds_messy_v2.jsonl"
if os.path.exists('real-eval/preds_messy_gemini.jsonl'):
    arms += " gemini=real-eval/preds_messy_gemini.jsonl"

!python real-eval/compare_arms.py --corpus real-eval/corpus_messy.jsonl {arms}

Add `--only-disagreements` to skip the lines every arm agrees on. The tail
prints per-field agreement and a **non-null count per field** — that second
table is where over-extraction shows up, so watch `latency_ms` and
`status_code`, the two fields v2 invented most on the labelled corpus.

Judge these by hand against `schema_v2.SPEC_EVAL`, not by which arm looks
confident:

- **`level` on lines that carry none.** `Error -60005 creating authorization`
  has no level field; `ERROR` there is prose. v2 and v3 both held 0/32 on
  gold-null in the labelled eval — check it survives on formats they have never
  seen.
- **`latency_ms` vs a duration in prose.** `took=4.775s` and `rt=7.881` are
  structural; `finished in 6.049 s` is not. This is the F2 family v3 was
  explicitly trained on.
- **`service` on the truncated line** (`<134>Jul 12 18:44:0`). There is no
  service token. Anything non-null is invention.
- **`timestamp` with no year.** The schema says a `1900` sentinel, not a guess
  at the current year — and several of these lines do carry a real year, so both
  behaviours should appear.

If v3 and Gemini disagree, decide which is right by reading the spec, then write
it down. Twenty hand-adjudicated messy lines is a small labelled corpus, and it
is worth more than any agreement percentage.

---

**Appendix — the synthetic result.** The v1 headline (100% vs 83.5% exact match
on a 200-example held-out set, `baseline.json` / README) was measured on data
from the same generator that produced the training set. It is reported for
completeness; the numbers above, on real logs from an entirely different
generative process, are the ones that mean anything.